# Soft-SPIN Healthcare — Step 3: generate answers (baseline + soft), swept over α

Generates the **soft** answers (coupled weights set to `α·W`) for the healthcare fairness (subset-tagged) and
privacy eval questions, for each α in the grid — the inputs to the Step-4 MD-Judge awareness scorer.
**Same generation protocol as the general soft/health Step 3:** greedy, `max_new_tokens=64`, raw questions;
originals cached once; for each α the coupled weights are reset to `α·W` (absolute assignment — soft scaling
is not idempotent); baseline generated (or reused) once with untouched weights. Checkpointed per (α, category)
→ resumable. α=0 = hard SPIN, α=1 = baseline (duplicates the baseline column — drop it to save a pass).

Awareness is near-saturated on healthcare (case study §4.4), so you will typically judge **one chosen α**
(the operating point from the Step-2 decoupling/capability curve). Generation is ~2 h per α on the full eval
(1,416 fairness + 657 privacy), so set `ALPHAS` to the point(s) you actually want to judge rather than the
whole grid — or run the grid across sessions (it resumes from the checkpoint).

**Inputs (Kaggle Dataset inputs):** `coupled_soft_health.json` (Soft Step-1), `eval_fair_health.csv`,
`eval_priv_health.csv` (general Step-2 splits), and optionally `answers_health_fairness.csv` /
`answers_health_privacy.csv` (to reuse the general-pipeline baseline column instead of regenerating it).
Internet ON. GPU **T4×2**. **Outputs:** `answers_health_soft_a{alpha}_{fairness,privacy}.csv`,
`results_health_soft_step3.json`.

## Config

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
import subprocess, sys, glob, json, time
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.46.3','sentencepiece'], check=False)
import pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID='Qwen/Qwen2-7B-Instruct'
USE_FAST=False
ALPHAS=[0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0]   # 0 = hard SPIN, 1 = baseline. Set to your chosen operating point(s) to save passes.
N_EVAL=None                # None = full eval sets; int to subsample (debug)
GEN_BS=4; MAXNEW=64; USE_CHAT_TEMPLATE=False
DTYPE=torch.float16
def find(name):
    for r in ['.','/kaggle/input','/kaggle/working']:
        h=glob.glob(os.path.join(r,'**',name),recursive=True)
        if h: return h[0]
    return None
print('model', MODEL_ID, '| alphas', ALPHAS)

## Coupled set + eval splits + baseline (reuse if attached, else generate)

In [ ]:
CP=find('coupled_soft_health.json'); assert CP, 'add coupled_soft_health.json (Soft Step-1 output) as input'
COUPLED={k:[int(x) for x in v] for k,v in json.load(open(CP))['coupled'].items()}
NZ=sum(len(v) for v in COUPLED.values())
print('coupled set:', CP, '| matrices', len(COUPLED), '| coupled weights', NZ)

EF=find('eval_fair_health.csv'); EP=find('eval_priv_health.csv')
assert EF and EP, 'add eval_fair_health.csv + eval_priv_health.csv (general Step-2 output) as input'
fair_df=pd.read_csv(EF); priv_df=pd.read_csv(EP)
if N_EVAL: fair_df=fair_df.iloc[:N_EVAL].copy(); priv_df=priv_df.iloc[:N_EVAL].copy()
if 'subset' not in fair_df.columns: fair_df['subset']='(all)'
fair_q=fair_df['question'].astype(str).tolist(); fair_sub=fair_df['subset'].astype(str).tolist()
priv_q=priv_df['question'].astype(str).tolist()
print('fairness', len(fair_q), '| privacy', len(priv_q))

def baseline_map(cat, qs):
    # reuse a 'baseline' column from an attached answers_health_*{cat}.csv if present; else {} -> generate
    for pat in [f'answers_health_{cat}.csv', f'answers_health_*{cat}.csv', f'answers_baseline_health_{cat}.csv']:
        f=find(pat)
        if not f: continue
        df=pd.read_csv(f)
        key='q' if 'q' in df.columns else ('question' if 'question' in df.columns else None)
        if 'baseline' in df.columns and key:
            print(f'  baseline {cat} <- {os.path.basename(f)} ({len(df)} rows)')
            return dict(zip(df[key].astype(str), df['baseline'].fillna('').astype(str)))
    print(f'  no reusable baseline for {cat} -> will GENERATE with the unsuppressed model'); return {}
BASE={'fairness':baseline_map('fairness',fair_q),'privacy':baseline_map('privacy',priv_q)}
gaps={c:sum(1 for q in (fair_q if c=='fairness' else priv_q) if str(q) not in BASE[c]) for c in BASE}
print('baseline coverage gaps (0 = fully reused; >0 = generated below):', gaps)

## Model + cache originals + generation / α-application helpers

In [ ]:
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID, use_fast=USE_FAST, trust_remote_code=True)
tokenizer.padding_side='left'
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
model=AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, trust_remote_code=True,
        device_map='auto', max_memory={0:'14GiB',1:'14GiB','cpu':'28GiB'}).eval()
GEN_DEV=model.get_input_embeddings().weight.device
print('model loaded | entry', GEN_DEV)

# cache ORIGINAL coupled-weight values ONCE (weights still untouched = baseline)
PARAMS=dict(model.named_parameters()); IDX={}; ORIG={}
for n,idxlist in COUPLED.items():
    if n in PARAMS and idxlist:
        p=PARAMS[n]; idx=torch.tensor(idxlist, device=p.device); IDX[n]=idx; ORIG[n]=p.data.view(-1)[idx].clone()
print('cached originals for', len(IDX), 'matrices,', int(sum(v.numel() for v in IDX.values())), 'weights')

def fmt(chunk):
    if USE_CHAT_TEMPLATE:
        return [tokenizer.apply_chat_template([{'role':'user','content':q}],tokenize=False,add_generation_prompt=True) for q in chunk]
    return [str(q) for q in chunk]
@torch.no_grad()
def apply_alpha(alpha):
    # absolute assignment: coupled weight <- alpha * original (idempotent across alpha values)
    nz=0
    for n,idx in IDX.items():
        PARAMS[n].data.view(-1)[idx]=alpha*ORIG[n]; nz+=idx.numel()
    return int(nz)
def restore_orig(): apply_alpha(1.0)
@torch.no_grad()
def gen_set(queries, ckkey, state, CK):
    done=state.get(ckkey,[]); t0=time.time()
    for i in range(len(done),len(queries),GEN_BS):
        enc=tokenizer(fmt(queries[i:i+GEN_BS]),return_tensors='pt',padding=True,truncation=True,max_length=256).to(GEN_DEV)
        g=model.generate(**enc,max_new_tokens=MAXNEW,do_sample=False,pad_token_id=tokenizer.pad_token_id)
        done+=[tokenizer.decode(g[j,enc.input_ids.shape[1]:],skip_special_tokens=True).strip() for j in range(g.shape[0])]
        state[ckkey]=done; json.dump(state,open(CK,'w'))
        print(f'    {ckkey} {len(done)}/{len(queries)} ({time.time()-t0:.0f}s)',end='\r')
    print(); return done
print('gen ready.')

## Baseline (untouched) once, then generate soft answers per α (checkpointed) + write CSVs

In [ ]:
CK='/kaggle/working/gen_soft_health_ckpt.json'
state=json.load(open(CK)) if os.path.exists(CK) else {}
# --- baseline (unsuppressed) only where a reused CSV did not fully cover it; BEFORE any apply_alpha ---
for cat,qs in [('fairness',fair_q),('privacy',priv_q)]:
    if sum(1 for q in qs if str(q) not in BASE[cat])==0: continue
    print(f'[baseline {cat}] generating with unsuppressed model ({len(qs)} q)')
    b=gen_set(qs, f'base_{cat}', state, CK)
    BASE[cat]=dict(zip([str(q) for q in qs], b))
    pd.DataFrame({'q':qs,'baseline':b}).to_csv(f'/kaggle/working/answers_baseline_health_{cat}.csv',index=False)

def changed(a,b): return int(sum(1 for x,y in zip(a,b) if str(x).strip()!=str(y).strip()))
def empties(a): return int(sum(1 for x in a if not str(x).strip()))
res={'model':MODEL_ID,'domain':'healthcare','variant':'soft-SPIN','coupled_count':NZ,'alphas':ALPHAS,
     'n_fair':len(fair_q),'n_priv':len(priv_q),'per_alpha':[]}
for a in ALPHAS:
    key=f'{a:.2f}'; nz=apply_alpha(a)
    print(f'alpha={key}: set {nz} coupled weights to alpha*W -> generating...')
    sf=gen_set(fair_q, f'a{key}_fairness', state, CK)
    sp=gen_set(priv_q, f'a{key}_privacy',  state, CK)
    bf=[BASE['fairness'].get(str(q),'') for q in fair_q]; bp=[BASE['privacy'].get(str(q),'') for q in priv_q]
    pd.DataFrame({'q':fair_q,'subset':fair_sub,'baseline':bf,'soft':sf}).to_csv(f'/kaggle/working/answers_health_soft_a{key}_fairness.csv',index=False)
    pd.DataFrame({'q':priv_q,'baseline':bp,'soft':sp}).to_csv(f'/kaggle/working/answers_health_soft_a{key}_privacy.csv',index=False)
    r_={'alpha':a,'coupled_scaled':int(nz),'changed_fair':changed(bf,sf),'changed_priv':changed(bp,sp),
        'empty_soft_fair':empties(sf),'empty_soft_priv':empties(sp)}
    res['per_alpha'].append(r_)
    print(f'  saved answers_health_soft_a{key}_*.csv | changed fair {r_["changed_fair"]}/{len(fair_q)} priv {r_["changed_priv"]}/{len(priv_q)} | empties fair {r_["empty_soft_fair"]} priv {r_["empty_soft_priv"]}')
restore_orig(); print('weights restored to originals.')
json.dump(res, open('/kaggle/working/results_health_soft_step3.json','w'), indent=2)
print('\nDONE for alphas', ALPHAS, '-> next: Step 4 MD-Judge (per subset + pooled) on answers_health_soft_a*_*.csv.')